# SangHyo CN / MCI / DEM 자동 실행 노트북

팀 공통 `base.ipynb`와 같은 실행 골조입니다. Colab에서 실행할 때마다 GitHub 저장소를 새로 clone하므로, 코드를 push한 뒤 **런타임 → 모두 실행**하면 최신 `Transformer + TabNet + Google YDF` 실험이 실행됩니다.

- GitHub의 최신 코드를 `/content/Google-Ajou-AICapstone`에 새로 받습니다.
- 공용 Google Drive의 `Data`를 읽습니다.
- Training-only EDA를 먼저 실행한 뒤 학습을 시작합니다.
- 모든 결과와 ZIP 압축본은 개인 Google Drive에 저장합니다.


## 셀 1. 기본 환경 준비

Colab에서는 Google Drive를 마운트하고 기존 임시 저장소를 삭제한 뒤 GitHub에서 최신 코드를 clone합니다. 로컬에서는 현재 저장소를 그대로 사용합니다.


In [ ]:
# Cell 1 - Environment setup
import json
import os
import runpy
import shutil
import subprocess
import sys
import traceback
from datetime import datetime
from pathlib import Path

REPO_URL = "https://github.com/Pig30nidaE/Google-Ajou-AICapstone.git"
REPO_DIR_NAME = "Google-Ajou-AICapstone"


def in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


def find_project_root():
    cwd = Path.cwd().resolve()
    for path in [cwd, *cwd.parents]:
        if (path / "base.ipynb").exists() or (path / "Data").exists():
            return path
    return None


IN_COLAB = in_colab()

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

    clone_path = Path("/content") / REPO_DIR_NAME
    os.chdir("/content")
    if clone_path.exists():
        print(f"Remove existing repo: {clone_path}")
        shutil.rmtree(clone_path)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(clone_path)],
        check=True,
    )
    PROJECT_ROOT = clone_path.resolve()
else:
    PROJECT_ROOT = find_project_root() or Path.cwd().resolve()

os.chdir(PROJECT_ROOT)

if IN_COLAB:
    DATA_ROOT = Path("/content/drive/Shareddrives/GoogleAI_contest/Data")
else:
    DATA_ROOT = PROJECT_ROOT / "Data"

if not DATA_ROOT.exists() and (PROJECT_ROOT / "Data").exists():
    DATA_ROOT = PROJECT_ROOT / "Data"

print(f"IN_COLAB     : {IN_COLAB}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_ROOT    : {DATA_ROOT}")


## 셀 2. 사용자 입력

일반적으로 `RUN_MODE`만 `smoke` 또는 `full`로 선택하면 됩니다. 정식 결과는 `full`입니다.


In [ ]:
# Cell 2 - User inputs
USER_FOLDER = "SangHyo"
EXPERIMENT_FOLDER = "ThreeClass_TransformerTabNet_Google"
RUN_FILE = "train.py"
EDA_FILE = "eda.py"

RUN_MODE = "full"  # 빠른 동작 확인: "smoke", 정식 실행: "full"
FEATURE_MODE = "clinical_plus_lifelog"  # 비교용: "wearable_only"
SEED = 20260719

# None이면 Colab은 MyDrive/SangHyo_CN_MCI_DEM_Results에 저장합니다.
RESULTS_ROOT_OVERRIDE = None


## 셀 3. 경로 확인

GitHub에서 받은 실험 코드, 공용 데이터, Google Drive 결과 경로를 확인합니다. 하나라도 없으면 학습 전에 바로 중단합니다.


In [ ]:
# Cell 3 - Resolve paths
USER_ROOT = (PROJECT_ROOT / USER_FOLDER).resolve()
EXPERIMENT_ROOT = (USER_ROOT / EXPERIMENT_FOLDER).resolve()
RUN_PATH = (EXPERIMENT_ROOT / RUN_FILE).resolve()
EDA_PATH = (EXPERIMENT_ROOT / EDA_FILE).resolve()
REQUIREMENTS_PATH = (EXPERIMENT_ROOT / "requirements_colab.txt").resolve()
TRAINING_ROOT = (DATA_ROOT / "1.Training").resolve()
VALIDATION_ROOT = (DATA_ROOT / "2.Validation").resolve()

if RESULTS_ROOT_OVERRIDE is not None:
    RESULTS_ROOT = Path(RESULTS_ROOT_OVERRIDE).expanduser().resolve()
elif IN_COLAB:
    RESULTS_ROOT = Path("/content/drive/MyDrive/SangHyo_CN_MCI_DEM_Results")
else:
    RESULTS_ROOT = EXPERIMENT_ROOT / "training_outputs"

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'.")
if FEATURE_MODE not in {"clinical_plus_lifelog", "wearable_only"}:
    raise ValueError("Unknown FEATURE_MODE.")

required_paths = {
    "user folder": USER_ROOT,
    "experiment folder": EXPERIMENT_ROOT,
    "training script": RUN_PATH,
    "EDA script": EDA_PATH,
    "requirements": REQUIREMENTS_PATH,
    "training data": TRAINING_ROOT,
    "validation data": VALIDATION_ROOT,
}
missing = [f"{name}: {path}" for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required paths:\n" + "\n".join(missing))

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = RESULTS_ROOT / f"{RUN_ID}_{RUN_MODE}_{FEATURE_MODE}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

for name, path in required_paths.items():
    print(f"{name:20s}: {path}")
print(f"{'output':20s}: {OUTPUT_DIR}")


## 셀 4. 실험 requirements 설치

실험 폴더의 `requirements_colab.txt`를 설치합니다. Colab에 포함된 CUDA용 PyTorch는 다시 설치하지 않습니다.


In [ ]:
# Cell 4 - Install experiment requirements
print(f"Installing: {REQUIREMENTS_PATH}")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REQUIREMENTS_PATH),
    ],
    check=True,
)
print("Requirements installation complete.")


## 셀 5. EDA와 학습 자동 실행

A100을 확인하고 Training-only EDA를 실행한 다음, Transformer·TabNet·Google YDF nested-CV 학습을 시작합니다. 끝나면 핵심 지표를 표시하고 전체 결과를 Google Drive ZIP으로 저장합니다.


In [ ]:
# Cell 5 - Run EDA and training
from importlib.metadata import version

import pandas as pd
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU가 없습니다. Colab 런타임 유형에서 A100 GPU를 선택해주세요."
    )

print(f"GPU            : {torch.cuda.get_device_name(0)}")
print(f"CUDA           : {torch.version.cuda}")
print(f"pytorch-tabnet : {version('pytorch-tabnet')}")
print(f"YDF            : {version('ydf')}")
print(f"Optuna         : {version('optuna')}")

EDA_OUTPUT_DIR = OUTPUT_DIR / "eda"
TRAINING_OUTPUT_DIR = OUTPUT_DIR / "training"

eda_command = [
    sys.executable,
    str(EDA_PATH),
    "--training-root",
    str(TRAINING_ROOT),
    "--output-dir",
    str(EDA_OUTPUT_DIR),
]

training_command = [
    sys.executable,
    str(RUN_PATH),
    "--training-root",
    str(TRAINING_ROOT),
    "--validation-root",
    str(VALIDATION_ROOT),
    "--output-dir",
    str(TRAINING_OUTPUT_DIR),
    "--feature-mode",
    FEATURE_MODE,
    "--outer-folds",
    "3",
    "--outer-repeats",
    "2",
    "--inner-folds",
    "3",
    "--trials-transformer",
    "24",
    "--trials-tabnet",
    "24",
    "--trials-ydf",
    "40",
    "--seed",
    str(SEED),
]
if RUN_MODE == "smoke":
    training_command.append("--fast")

def run_python_file(script_path, arguments):
    """공통 base.ipynb처럼 현재 Colab 커널에서 파일을 실행합니다."""
    previous_cwd = Path.cwd()
    previous_argv = sys.argv[:]
    script_dir = str(script_path.parent)
    inserted_path = script_dir not in sys.path
    if inserted_path:
        sys.path.insert(0, script_dir)
    os.chdir(script_path.parent)
    sys.argv = [str(script_path), *arguments]
    try:
        return runpy.run_path(str(script_path), run_name="__main__")
    finally:
        sys.argv = previous_argv
        os.chdir(previous_cwd)
        if inserted_path and script_dir in sys.path:
            sys.path.remove(script_dir)


try:
    print("\n[1/2] Training-only EDA")
    print(" ".join(eda_command))
    run_python_file(EDA_PATH, eda_command[2:])

    print("\n[2/2] Model training")
    print(" ".join(training_command))
    run_python_file(RUN_PATH, training_command[2:])
except BaseException:
    failure_text = traceback.format_exc()
    failure_path = OUTPUT_DIR / "FAILED_TRACEBACK.log"
    failure_path.write_text(failure_text, encoding="utf-8")
    print("\n실제 오류 traceback:\n")
    print(failure_text)
    print(f"오류 로그 저장: {failure_path}")
    raise

report_path = TRAINING_OUTPUT_DIR / "FINAL_REPORT.json"
with report_path.open(encoding="utf-8") as handle:
    report = json.load(handle)

nested = report["nested_oof"]
validation = report.get("validation")
metric_names = ["accuracy", "macro_f1", "roc_auc_ovr_macro", "balanced_accuracy"]
summary_rows = [
    {"split": "nested OOF", **{name: nested[name] for name in metric_names}}
]
if validation is not None:
    summary_rows.append(
        {"split": "validation", **{name: validation[name] for name in metric_names}}
    )
display(pd.DataFrame(summary_rows))
print("Target check:", report["target_check_nested_oof"])

archive_base = RESULTS_ROOT / f"{OUTPUT_DIR.name}_archive"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR)
print(f"Result folder : {OUTPUT_DIR}")
print(f"Result archive: {archive_path}")
print("Done.")
